# Building a RAG System — Part 1: Ingestion Pipeline

**Internal Ship Program · Tasks 2.1 → 2.4**

Welcome! This notebook teaches the **ingestion** half of a Retrieval-Augmented Generation (RAG)
system, hands-on, using the **LangChain ecosystem** and **local open-source models** (no API keys,
your data never leaves the machine).

We process a real document — the **CIS Controls v8** security guidelines (`data/`) — through four stages:

```
        ┌──────────┐    ┌──────────┐    ┌───────────┐    ┌──────────────┐
  PDF → │ 2.1 PARSE│ →  │2.2 CHUNK │ →  │2.3 EMBED  │ →  │2.4 STORE     │ → (later: retrieve → rerank → agent)
        │unstructured   │splitters │    │bge-small  │    │Weaviate      │
        └──────────┘    └──────────┘    └───────────┘    └──────────────┘
```

| Stage | What it does | Tool |
|-------|--------------|------|
| **2.1 Parse** | Turn a messy PDF into clean text + structure (titles, tables, images) | `langchain-unstructured` |
| **2.2 Chunk** | Split text into retrieval-sized pieces | `langchain-text-splitters` + unstructured |
| **2.3 Embed** | Turn each chunk into a vector (numbers that capture meaning) | `langchain-huggingface` (`bge-small-en-v1.5`) |
| **2.4 Store** | Save vectors in a database we can search by similarity | `langchain-weaviate` |

> **Why RAG?** LLMs don't know your private docs. RAG lets the model *retrieve* relevant chunks
> from your documents at question time and answer grounded in them. Everything here is the
> "prepare the knowledge" half — retrieval, reranking, and the agent come next"


## 0. Setup

### Python dependencies
Already declared in `pyproject.toml`. If you cloned fresh, run **`uv sync`** in a terminal, then
select this project's `.venv` as the notebook kernel. (Or run the cell below to install on the fly.)

### System packages (required for hi-res PDF parsing)
`unstructured` uses OCR + a layout model to detect tables and images. On Debian/Ubuntu/WSL:

```bash
sudo apt-get update && sudo apt-get install -y poppler-utils tesseract-ocr libgl1
```

- **poppler-utils** → renders PDF pages to images
- **tesseract-ocr** → reads text from those images (OCR)
- **libgl1** → shared library the layout/vision model needs

> First run downloads the embedding model (~130 MB) and the layout model. Be patient once; they're cached after.


In [1]:
import sys

print(sys.version)
print(sys.executable)

3.12.13 (main, Jun 23 2026, 15:23:43) [MSC v.1944 64 bit (AMD64)]
C:\Users\User\Desktop\DAR\rag_setup\rag_setup\.venv\Scripts\python.exe


In [3]:
import langchain
import langchain_huggingface
import langchain_text_splitters
import langchain_unstructured
import langchain_weaviate
import sentence_transformers
import unstructured
import weaviate

print("All RAG dependencies imported successfully.")

All RAG dependencies imported successfully.


In [1]:
import sys

print(sys.version)
print(sys.executable)

3.12.13 (main, Jun 23 2026, 15:23:43) [MSC v.1944 64 bit (AMD64)]
C:\Users\User\Desktop\DAR\rag_setup\rag_setup\.venv\Scripts\python.exe


In [2]:
from pathlib import Path

filename = "CIS_Controls__v8__Critical_Security_Controls__2023_08.pdf"

possible_paths = [
    Path.cwd() / "data" / filename,
    Path.cwd().parent / "data" / filename,
]

pdf_path = next(
    (path.resolve() for path in possible_paths if path.exists()),
    None
)

print("Current directory:", Path.cwd())
print("PDF path:", pdf_path)
print("PDF found:", pdf_path is not None)

Current directory: C:\Users\User\Desktop\DAR\rag_setup\rag_setup\notebooks
PDF path: C:\Users\User\Desktop\DAR\rag_setup\rag_setup\data\CIS_Controls__v8__Critical_Security_Controls__2023_08.pdf
PDF found: True


In [5]:
import os
import shutil
from pathlib import Path

tesseract_dir = Path(r"C:\Program Files\Tesseract-OCR")
tesseract_exe = tesseract_dir / "tesseract.exe"

if not tesseract_exe.exists():
    raise FileNotFoundError(f"Tesseract not found at: {tesseract_exe}")

# Make Tesseract visible to this Jupyter kernel
os.environ["PATH"] = (
    str(tesseract_dir)
    + os.pathsep
    + os.environ.get("PATH", "")
)

# Explicitly configure the package used by Unstructured
import unstructured_pytesseract.pytesseract as pytesseract

pytesseract.tesseract_cmd = str(tesseract_exe)

print("Tesseract executable:", shutil.which("tesseract"))
print("Configured path:", pytesseract.tesseract_cmd)

Tesseract executable: C:\Program Files\Tesseract-OCR\tesseract.EXE
Configured path: C:\Program Files\Tesseract-OCR\tesseract.exe


In [6]:
import subprocess

result = subprocess.run(
    ["tesseract", "--version"],
    capture_output=True,
    text=True,
    check=True,
)

print(result.stdout.splitlines()[0])

tesseract v5.4.0.20240606


In [7]:
from langchain_unstructured import UnstructuredLoader

loader = UnstructuredLoader(
    file_path=str(pdf_path),
    partition_via_api=False,
    strategy="hi_res",
    languages=["eng"],
    infer_table_structure=True,
)

print("Parser configured successfully.")

Parser configured successfully.


In [8]:
import time

print("Parsing started...")
start_time = time.perf_counter()

documents = loader.load()

elapsed = time.perf_counter() - start_time

print(f"Parsing completed in {elapsed:.1f} seconds")
print(f"Number of parsed elements: {len(documents)}")

Parsing started...


INFO: Reading PDF for file: C:\Users\User\Desktop\DAR\rag_setup\rag_setup\data\CIS_Controls__v8__Critical_Security_Controls__2023_08.pdf ...
INFO: Loading the Table agent ...
INFO: HTTP Request: HEAD https://huggingface.co/microsoft/table-transformer-structure-recognition/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
INFO: HTTP Request: HEAD https://huggingface.co/microsoft/table-transformer-structure-recognition/resolve/main/preprocessor_config.json "HTTP/1.1 307 Temporary Redirect"
INFO: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/microsoft/table-transformer-structure-recognition/f4d4bdc85c3fe4b1fa49658882a5d38bbdd0f343/preprocessor_config.json "HTTP/1.1 200 OK"
INFO: HTTP Request: GET https://huggingface.co/api/resolve-cache/models/microsoft/table-transformer-structure-recognition/f4d4bdc85c3fe4b1fa49658882a5d38bbdd0f343/preprocessor_config.json "HTTP/1.1 200 OK"


preprocessor_config.json:   0%|          | 0.00/274 [00:00<?, ?B/s]

INFO: Loading table structure model to cpu...
INFO: HTTP Request: HEAD https://huggingface.co/microsoft/table-transformer-structure-recognition/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/microsoft/table-transformer-structure-recognition/f4d4bdc85c3fe4b1fa49658882a5d38bbdd0f343/config.json "HTTP/1.1 200 OK"
INFO: HTTP Request: GET https://huggingface.co/api/resolve-cache/models/microsoft/table-transformer-structure-recognition/f4d4bdc85c3fe4b1fa49658882a5d38bbdd0f343/config.json "HTTP/1.1 200 OK"


config.json:   0%|          | 0.00/1.47k [00:00<?, ?B/s]

INFO: HTTP Request: HEAD https://huggingface.co/microsoft/table-transformer-structure-recognition/resolve/main/model.safetensors "HTTP/1.1 302 Found"
INFO: HTTP Request: GET https://huggingface.co/api/models/microsoft/table-transformer-structure-recognition/xet-read-token/f4d4bdc85c3fe4b1fa49658882a5d38bbdd0f343 "HTTP/1.1 200 OK"


model.safetensors:   0%|          | 0.00/115M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/367 [00:00<?, ?it/s]

INFO: Table model successfully loaded to cpu


Parsing completed in 740.4 seconds
Number of parsed elements: 2655


In [9]:
from collections import Counter

categories = Counter(
    document.metadata.get("category", "Unknown")
    for document in documents
)

print("Total elements:", len(documents))
print("\nElements by category:")

for category, count in categories.most_common():
    print(f"{category}: {count}")

Total elements: 2655

Elements by category:
UncategorizedText: 1487
NarrativeText: 636
Title: 318
ListItem: 147
Table: 50
Image: 16
Header: 1


In [14]:
clean_documents = [
    document
    for document in documents
    if (
        document.page_content.strip()
        and (
            document.metadata.get("category")
            in {"Title", "NarrativeText", "ListItem", "Table"}
            or (
                document.metadata.get("category") == "UncategorizedText"
                and len(document.page_content.strip()) >= 30
            )
        )
    )
]

print("Original elements:", len(documents))
print("Clean elements:", len(clean_documents))
print("Removed elements:", len(documents) - len(clean_documents))

Original elements: 2655
Clean elements: 1248
Removed elements: 1407
